In [1]:
import numpy as np
import pandas as pd
import statsmodels.api as sm

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

/Users/isabellacaruz/anaconda3/lib/python3.11/site-packages/pandas/core/arrays/masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


# Homework 3: Linear Regression

In [2]:
train_df = pd.read_csv("train.csv")
test_df  = pd.read_csv("test.csv")

drop_cols = ["id", "date", "zipcode"]
train_df = train_df.drop(columns=drop_cols, errors="ignore").copy()
test_df  = test_df.drop(columns=drop_cols, errors="ignore").copy()

train_df["price"] = train_df["price"] / 1000
test_df["price"]  = test_df["price"] / 1000

X_train = train_df.drop(columns=["price"])
y_train = train_df["price"].astype(float)

X_test = test_df.drop(columns=["price"])
y_test = test_df["price"].astype(float)

scaler = StandardScaler()

X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns
)

X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns
)

In [3]:
X_train_sm = sm.add_constant(X_train_scaled) 
X_test_sm  = sm.add_constant(X_test_scaled)

model = sm.OLS(y_train, X_train_sm).fit()

print(model.summary())



                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       0.727
Model:                            OLS   Adj. R-squared:                  0.722
Method:                 Least Squares   F-statistic:                     153.9
Date:                Fri, 13 Feb 2026   Prob (F-statistic):          1.18e-262
Time:                        19:33:27   Log-Likelihood:                -6596.5
No. Observations:                1000   AIC:                         1.323e+04
Df Residuals:                     982   BIC:                         1.332e+04
Df Model:                          17                                         
Covariance Type:            nonrobust                                         
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const           520.4148      5.656     92.009

In [4]:
train_pred = model.predict(X_train_sm)
train_mse = mean_squared_error(y_train, train_pred)
train_r2  = r2_score(y_train, train_pred)

print("Training MSE:", train_mse)
print("Training R^2:", train_r2)


Training MSE: 31415.747916100874
Training R^2: 0.7271450489303787


In [5]:
test_pred = model.predict(X_test_sm)

test_mse = mean_squared_error(y_test, test_pred)
test_r2  = r2_score(y_test, test_pred)

print("Testing MSE:", test_mse)
print("Testing R^2:", test_r2)

Testing MSE: 58834.673978213905
Testing R^2: 0.6471195893437878


In [6]:
ols_model = sm.OLS(y_train, X_train_sm).fit()
ols_model.params.drop("const").sort_values(ascending=False).head(5)

grade          92.511076
lat            78.129852
waterfront     64.230911
sqft_living    57.161582
sqft_above     48.439051
dtype: float64

## [C] Problem 3:  Implementing closed-form solution for linear regression (15 points) 

In [7]:

X_train_np = X_train_scaled.to_numpy()
X_test_np  = X_test_scaled.to_numpy()

y_train_np = y_train.to_numpy()
y_test_np  = y_test.to_numpy()

X_train_np = np.c_[np.ones((X_train_np.shape[0], 1)), X_train_np]
X_test_np  = np.c_[np.ones((X_test_np.shape[0], 1)), X_test_np]

theta = np.linalg.pinv(X_train_np) @ y_train_np

train_pred_cf = X_train_np @ theta
test_pred_cf  = X_test_np @ theta

print("Closed-form Training MSE:", mean_squared_error(y_train_np, train_pred_cf))
print("Closed-form Training R²:", r2_score(y_train_np, train_pred_cf))

print("Closed-form Testing MSE:", mean_squared_error(y_test_np, test_pred_cf))
print("Closed-form Testing R²:", r2_score(y_test_np, test_pred_cf))


Closed-form Training MSE: 31415.747916100874
Closed-form Training R²: 0.7271450489303787
Closed-form Testing MSE: 58834.673978213905
Closed-form Testing R²: 0.6471195893437878


## [C] Problem 4: Polynomial Regression (15 points) 

In [8]:
X_train_sqft = train_df["sqft_living"].to_numpy()
X_test_sqft  = test_df["sqft_living"].to_numpy()

y_train_np = y_train.to_numpy()
y_test_np  = y_test.to_numpy()


In [9]:
def make_poly_feature(x, degree):
    return np.column_stack([x**i for i in range(1, degree+1)])

In [ ]:
degrees = [1, 2, 3, 5]

for p in degrees:

    Xtr_poly = make_poly_feature(X_train_sqft, p)
    Xte_poly = make_poly_feature(X_test_sqft, p)

    Xtr_poly = np.c_[np.ones(len(Xtr_poly)), Xtr_poly]
    Xte_poly = np.c_[np.ones(len(Xte_poly)), Xte_poly]

    theta = np.linalg.pinv(Xtr_poly) @ y_train_np

    train_pred = Xtr_poly @ theta
    test_pred  = Xte_poly @ theta

    print(f"\np = {p}")
    print("Train MSE:", mean_squared_error(y_train_np, train_pred))
    print("Train R²:", r2_score(y_train_np, train_pred))
    print("Test MSE:", mean_squared_error(y_test_np, test_pred))
    print("Test R²:", r2_score(y_test_np, test_pred))



p = 1
Train MSE: 57947.52616128836
Train R²: 0.49670880166311404
Test MSE: 88575.97854309603
Test R²: 0.46873628136126433

p = 2
Train MSE: 54822.66511627667
Train R²: 0.5238491329967203
Test MSE: 71791.67947889882
Test R²: 0.5694056646664865

p = 3
Train MSE: 53785.194716493934
Train R²: 0.5328598665920145
Test MSE: 99833.48320573066
Test R²: 0.4012156748939668

p = 5
Train MSE: 54114.912599428826
Train R²: 0.5299961704274272
Test MSE: 93265064.78684787
Test R²: -558.3880638150217


## [C] Problem 5:  Gradient descent (20 points) 

In [11]:
X_train_np = X_train_scaled.to_numpy()
X_test_np  = X_test_scaled.to_numpy()

y_train_np = y_train.to_numpy()
y_test_np  = y_test.to_numpy()

X_train_gd = np.c_[np.ones((X_train_np.shape[0], 1)), X_train_np]
X_test_gd  = np.c_[np.ones((X_test_np.shape[0], 1)),  X_test_np]

In [13]:
alphas = [0.01, 0.1, 0.5]
iters_list = [10, 50, 100]
rows = []

for a in alphas:
    for it in iters_list:
        theta = np.zeros(X_train_gd.shape[1])

        for _ in range(it):
            grad = (1/len(y_train)) * (X_train_gd.T @ (X_train_gd @ theta - y_train))
            theta = theta - a * grad

        train_pred = X_train_gd @ theta
        test_pred  = X_test_gd @ theta

        rows.append({
            "alpha": a,
            "iters": it,
            "Train MSE": mean_squared_error(y_train, train_pred),
            "Train R^2": r2_score(y_train, train_pred),
            "Test MSE": mean_squared_error(y_test, test_pred),
            "Test R^2": r2_score(y_test, test_pred),
        })

pd.DataFrame(rows)

,alpha,iters,Train MSE,Train R^2,Test MSE,Test R^2
0,0.01,10,2.947967e+05,-1.560395e+00,3.524490e+05,-1.113929e+00
1,0.01,50,1.382985e+05,-2.011631e-01,1.704509e+05,-2.233580e-02
2,0.01,100,7.009438e+04,3.912098e-01,9.475122e+04,4.316983e-01
3,0.10,10,6.647361e+04,4.226573e-01,9.087301e+04,4.549591e-01
4,0.10,50,3.151072e+04,7.263202e-01,5.892963e+04,6.465501e-01
5,0.10,100,3.142750e+04,7.270429e-01,5.889174e+04,6.467773e-01
6,0.50,10,6.163610e+08,-5.352276e+03,6.888098e+08,-4.130365e+03
7,0.50,50,1.708464e+25,-1.483851e+20,1.904481e+25,-1.142275e+20
8,0.50,100,6.110787e+45,-5.307397e+40,6.811893e+45,-4.085658e+40


## [A/C] Problem 6: Ridge regularization (20 points) 

In [14]:
def gd_ridge(X, y, alpha, iters, lam):
    n, d = X.shape
    theta = np.zeros(d)

    for _ in range(iters):
        grad = (1/n) * (X.T @ (X @ theta - y))

        ridge_grad = (2 * lam / n) * theta
        ridge_grad[0] = 0.0

        theta = theta - alpha * (grad + ridge_grad)

    return theta


In [15]:
lambdas = [1, 10, 100, 1000, 10000]

for lam in lambdas:
    theta = gd_ridge(X_train_gd, y_train_np, alpha=0.1, iters=100, lam=lam)

    train_pred = X_train_gd @ theta
    test_pred  = X_test_gd @ theta

    print("λ =", lam)
    print("Slope:", round(theta[1], 4))
    print("Train MSE:", mean_squared_error(y_train_np, train_pred))
    print("Test MSE:", mean_squared_error(y_test_np, test_pred))
    print("Train R²:", r2_score(y_train_np, train_pred))
    print("Test R²:", r2_score(y_test_np, test_pred))
    print()


λ = 1
Slope: 8.3184
Train MSE: 31428.61569383944
Test MSE: 58898.63013468582
Train R²: 0.7270332885204529
Test R²: 0.6467359911484759

λ = 10
Slope: 8.0235
Train MSE: 31447.010063835874
Test MSE: 58970.9014251895
Train R²: 0.7268735280417675
Test R²: 0.6463025201874397

λ = 100
Slope: 5.7646
Train MSE: 32172.420211456425
Test MSE: 60336.022409093755
Train R²: 0.720573128927826
Test R²: 0.6381147557141644

λ = 1000
Slope: 0.412
Train MSE: 46339.74115676204
Test MSE: 80356.48321187693
Train R²: 0.5975258064944227
Test R²: 0.5180354223566144

λ = 10000
Slope: 5.2705465931324166e+17
Train MSE: 5.47108247528124e+39
Test MSE: 6.098793795515917e+39
Train R²: -4.751795007642154e+34
Test R²: -3.6579532332687717e+34



In [ ]:

np.random.seed(0)

N = 1000
X = np.random.uniform(-2, 2, size=N)
e = np.random.normal(0, np.sqrt(2), size=N)   
y = 1 + 2*X + e

Xmat = np.c_[np.ones(N), X]  

theta_ols = np.linalg.pinv(Xmat) @ y
yhat_ols = Xmat @ theta_ols

rows = []
rows.append({
    "Model": "OLS",
    "lambda": 0,
    "intercept": theta_ols[0],
    "slope": theta_ols[1],
    "MSE": mean_squared_error(y, yhat_ols),
    "R^2": r2_score(y, yhat_ols)
})

lambdas = [1, 10, 100, 1000, 10000]
R = np.diag([0.0, 1.0])  

for lam in lambdas:
    theta_ridge = np.linalg.inv(Xmat.T @ Xmat + lam * R) @ (Xmat.T @ y)
    yhat = Xmat @ theta_ridge

    rows.append({
        "Model": "Ridge",
        "lambda": lam,
        "intercept": theta_ridge[0],
        "slope": theta_ridge[1],
        "MSE": mean_squared_error(y, yhat),
        "R^2": r2_score(y, yhat)
    })

results = pd.DataFrame(rows)
results


,Model,lambda,intercept,slope,MSE,R^2
0,OLS,0,1.040515,1.965692,1.865374,0.736759
1,Ridge,1,1.040491,1.964238,1.865377,0.736759
2,Ridge,10,1.040279,1.951251,1.865656,0.736720
3,Ridge,100,1.038305,1.830236,1.890166,0.733261
4,Ridge,1000,1.026876,1.129641,2.809812,0.603481
5,Ridge,10000,1.012264,0.233982,5.917267,0.164958
